# Academic comparison of developmental-concern prediction models

This notebook consolidates the repository's **aggregate-only** evaluation evidence for the three independently modelled outcomes: Communication, Cognitive, and Social-emotional concern. It compares the original selected models, later individual candidates, classical hybrids, and TabICLv2-related experiments without treating any single metric as proof of overall superiority.

**Research boundary.** The notebook reads generated JSON reports only. It does not open participant-level datasets, fit models, adjust thresholds, write predictions, or expose identifiers. The overall concern decision is outside these tables because it is derived by OR of the three domain decisions rather than trained as a fourth model.

## Interpretation protocol

1. Development results and secondary-cohort results are presented separately.
2. PR-AUC is the primary ranking measure because positive outcomes are uncommon; higher is better.
3. ROC-AUC, balanced accuracy, sensitivity, specificity, precision and F1 are supporting measures; higher is better. Brier score assesses probabilistic error; lower is better.
4. A point estimate does not establish superiority. Paired uncertainty is shown where the stored report supports it.
5. Repeated-development intervals are descriptive because repeats share participants.
6. The reused 125-participant cohort provides secondary exploratory evidence, not fresh independent confirmation.
7. No clinical or diagnostic interpretation should be made from these research comparisons.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)
plt.style.use('seaborn-v0_8-whitegrid')

def repository_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'reports' / 'generated').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from the repository or its notebooks directory.')

ROOT = repository_root()
REPORTS = ROOT / 'reports' / 'generated'

def load_report(name):
    with (REPORTS / name).open(encoding='utf-8') as handle:
        return json.load(handle)

DOMAINS = {
    'target_communication_concern': 'Communication',
    'target_cognitive_concern': 'Cognitive',
    'target_social_emotional_concern': 'Social-emotional',
}
METRICS = ['average_precision_pr_auc', 'roc_auc', 'balanced_accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'brier_score']
DISPLAY_METRICS = {
    'average_precision_pr_auc': 'PR-AUC', 'roc_auc': 'ROC-AUC',
    'balanced_accuracy': 'Balanced accuracy', 'sensitivity': 'Sensitivity',
    'specificity': 'Specificity', 'precision': 'Precision', 'f1': 'F1',
    'brier_score': 'Brier score',
}
print(f'Aggregate report directory: {REPORTS}')

## 1. Evidence and comparability audit

The audit below makes the evaluation stage explicit before numerical comparisons are shown. Models should only be directly ranked when their estimates arise from a sufficiently comparable evaluation design.

In [ ]:
evidence = pd.DataFrame([
    ['Original five individual candidates', 'Development', 500, 'Repeated 5×5 participant-safe CV', 'Yes, within shared original folds', 'development-model-comparison-report.json'],
    ['CatBoost and EBM', 'Development', 500, 'Repeated 5×5 participant-safe nested CV', 'Shared with candidates in its report', 'development-additional-model-comparison-report.json'],
    ['LightGBM', 'Development', 500, 'Repeated 5×5 participant-safe nested CV', 'Numerically aligned; no paired uncertainty', 'development-lightgbm-comparison-report.json'],
    ['FIGS', 'Development', 500, 'Repeated 5×5 participant-safe nested CV', 'Numerically aligned; no paired uncertainty', 'development-figs-comparison-report.json'],
    ['HistGradientBoosting', 'Development', 500, 'Repeated 5×5 participant-safe nested CV', 'Numerically aligned; no paired uncertainty', 'development-histgradientboosting-comparison-report.json'],
    ['TabICLv2', 'Development', 500, 'Repeated 5×5 participant-safe nested CV', 'Numerically aligned; no paired uncertainty', 'development-tabiclv2-model-comparison-report.json'],
    ['Logistic–XGBoost hybrids', 'Development', 500, 'Repeated 5×5 participant-safe nested CV', 'Yes, within the hybrid experiment', 'development-hybrid-model-comparison-report.json'],
    ['Frozen selected models', 'Original locked test', 125, 'One-time frozen evaluation', 'Original confirmatory use', 'final-locked-test-report.json'],
    ['Equal Logistic–XGBoost hybrid', 'Secondary exploratory', 125, 'Reused previously evaluated cohort', 'Paired with original in report', 'secondary-exploratory-hybrid-evaluation-report.json'],
    ['TabICLv2', 'Secondary exploratory', 125, 'Reused previously evaluated cohort', 'Paired with original in report', 'secondary-tabiclv2-test-evaluation-report.json'],
], columns=['Evidence', 'Stage', 'Participants', 'Design', 'Comparison status', 'Source report'])
display(evidence)

## 2. Development comparison of individual model families

This table includes all eleven individual families represented by completed aggregate reports: Logistic Regression, RBF SVM, Random Forest, Extra Trees, XGBoost, CatBoost, EBM, LightGBM, FIGS, HistGradientBoosting and TabICLv2. Values are participant-averaged out-of-fold metrics where available, or the corresponding mean repeated-validation summary for the isolated later candidates.

In [ ]:
NAME = {
    'logistic_l2_balanced': 'Logistic Regression',
    'svm_rbf_balanced': 'RBF SVM',
    'random_forest_balanced_leaf5': 'Random Forest',
    'extra_trees_balanced_leaf5': 'Extra Trees',
    'xgboost_regularized_balanced': 'XGBoost',
    'catboost_ordered_balanced': 'CatBoost',
    'ebm_main_effects_balanced': 'EBM',
    'lightgbm_fixed': 'LightGBM',
    'figs_fixed': 'FIGS',
    'histgradientboosting_fixed': 'HistGradientBoosting',
    'tabiclv2_fixed': 'TabICLv2',
}
rows = []
base = load_report('development-additional-model-comparison-report.json')
for target, domain in DOMAINS.items():
    for key, result in base['domains'][target]['candidates'].items():
        metrics = result['averaged_oof_metrics']
        rows.append({'Domain': domain, 'Model': NAME.get(key, key), 'Experiment': 'Shared original/additional comparison', **{m: metrics[m] for m in METRICS}})

isolated = [
    ('development-lightgbm-comparison-report.json', 'LightGBM'),
    ('development-figs-comparison-report.json', 'FIGS'),
    ('development-histgradientboosting-comparison-report.json', 'HistGradientBoosting'),
    ('development-tabiclv2-model-comparison-report.json', 'TabICLv2'),
]
for filename, model in isolated:
    report = load_report(filename)
    for target, domain in DOMAINS.items():
        summary = report['domains'][target]['repeat_metric_summary']
        rows.append({'Domain': domain, 'Model': model, 'Experiment': 'Later isolated development comparison', **{m: summary[m]['mean'] for m in METRICS}})

development = pd.DataFrame(rows)
development = development.rename(columns=DISPLAY_METRICS)
development['PR-AUC rank'] = development.groupby('Domain')['PR-AUC'].rank(method='min', ascending=False).astype(int)
development = development.sort_values(['Domain', 'PR-AUC rank', 'Model'])
display(development[['Domain', 'Model', 'Experiment', 'PR-AUC rank', *DISPLAY_METRICS.values()]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True)
for ax, domain in zip(axes, DOMAINS.values()):
    view = development[development['Domain'] == domain].sort_values('PR-AUC')
    colors = ['#169BD5' if model != 'TabICLv2' else '#F44B93' for model in view['Model']]
    ax.barh(view['Model'], view['PR-AUC'], color=colors)
    ax.set_title(domain)
    ax.set_xlabel('Development PR-AUC (higher is better)')
    ax.set_xlim(0, 1)
fig.suptitle('Individual model families: descriptive development comparison', fontsize=15)
fig.tight_layout()
plt.show()

**Caution:** the PR-AUC rank is a descriptive ordering of point estimates, not a statistical claim. Some later isolated reports do not contain aligned participant-level predictions against every earlier candidate, so a universal paired significance test cannot be reconstructed from aggregate reports.

## 3. Development comparison of Logistic–XGBoost hybrids

This experiment compares both component models, equal-weight soft voting and nested-CV selected-weight soft voting on identical outer observations. It is evaluated separately from the individual-family table to preserve the experiment boundary.

In [ ]:
hybrid_report = load_report('development-hybrid-model-comparison-report.json')
hybrid_names = {
    'logistic_l2_balanced': 'Logistic Regression',
    'xgboost_regularized_balanced': 'XGBoost',
    'equal_weight_soft_voting': 'Equal-weight Logistic–XGBoost',
    'selected_weight_soft_voting': 'Selected-weight Logistic–XGBoost',
}
hybrid_rows, hybrid_ci_rows = [], []
for target, domain in DOMAINS.items():
    result = hybrid_report['domains'][target]
    for key, method in result['methods'].items():
        metrics = method['averaged_oof_metrics']
        hybrid_rows.append({'Domain': domain, 'Method': hybrid_names[key], **{m: metrics[m] for m in METRICS}})
    uncertainty = result['paired_uncertainty_selected_hybrid_vs_existing_reference']['differences']
    for metric, values in uncertainty.items():
        hybrid_ci_rows.append({'Domain': domain, 'Contrast': 'Selected-weight hybrid minus existing selected model', 'Metric': DISPLAY_METRICS.get(metric, metric), 'Mean difference': values['mean'], 'CI lower': values['confidence_interval_95'][0], 'CI upper': values['confidence_interval_95'][1]})

hybrid_development = pd.DataFrame(hybrid_rows).rename(columns=DISPLAY_METRICS)
display(hybrid_development.round(3))
display(pd.DataFrame(hybrid_ci_rows).round(3))

## 4. Supervisor-ready comparison on the common 125-participant cohort

This is the clearest direct comparison for presentation because the previously selected individual model, equal-weight Logistic–XGBoost hybrid, and TabICLv2 were evaluated on the same 125 participants. Within each domain, rows are ordered from highest to lowest **balanced accuracy**. PR-AUC and sensitivity are retained so that ranking quality and the ability to identify observed concerns are not hidden by the ordering rule.

This remains **secondary exploratory evidence**, not a fresh independent test, because the cohort had already been evaluated before the later methods were assessed. TabICLv2 is therefore labelled as a later experimental candidate, not a conclusively superior final model.

In [ ]:
secondary_hybrid = load_report('secondary-exploratory-hybrid-evaluation-report.json')
secondary_tabicl = load_report('secondary-tabiclv2-test-evaluation-report.json')
secondary_rows = []
for target, domain in DOMAINS.items():
    h = secondary_hybrid['domains'][target]
    t = secondary_tabicl['domains'][target]
    original_version = h['original_individual']['model_configuration_version'].lower()
    original_name = 'XGBoost' if 'xgboost' in original_version else 'Logistic Regression'
    methods = [
        ('Initially selected', original_name, h['original_individual']['metrics']),
        ('Hybrid candidate', 'Equal-weight Logistic–XGBoost', h['equal_weight_hybrid']['metrics']),
        ('Later experimental candidate', 'TabICLv2', t['tabiclv2']['metrics']),
    ]
    for stage, model, metrics in methods:
        secondary_rows.append({
            'Domain': domain, 'Model stage': stage, 'Model': model,
            'PR-AUC': metrics['average_precision_pr_auc'],
            'Balanced accuracy': metrics['balanced_accuracy'],
            'Sensitivity': metrics['sensitivity'],
            'Evidence status': 'Secondary exploratory — reused cohort',
        })
secondary = pd.DataFrame(secondary_rows)
secondary['Balanced-accuracy rank'] = secondary.groupby('Domain')['Balanced accuracy'].rank(method='min', ascending=False).astype(int)
secondary = secondary.sort_values(['Domain', 'Balanced-accuracy rank', 'Model'])
supervisor_table = secondary[['Domain', 'Balanced-accuracy rank', 'Model stage', 'Model', 'PR-AUC', 'Balanced accuracy', 'Sensitivity', 'Evidence status']]
display(supervisor_table.round(3).style.hide(axis='index').format({'PR-AUC': '{:.3f}', 'Balanced accuracy': '{:.3f}', 'Sensitivity': '{:.3f}'}))

In [ ]:
secondary_ci = []
for target, domain in DOMAINS.items():
    h = secondary_hybrid['domains'][target]
    t = secondary_tabicl['domains'][target]
    for metric in ['average_precision_pr_auc', 'balanced_accuracy', 'sensitivity']:
        h_ci = h['paired_uncertainty']['hybrid_minus_original_95_percentile_intervals'][metric]
        t_ci = t['paired_uncertainty']['tabiclv2_minus_original_95_percentile_intervals'][metric]
        secondary_ci.extend([
            {'Domain': domain, 'Contrast': 'Equal hybrid minus original', 'Metric': DISPLAY_METRICS[metric], 'Point difference': h['hybrid_minus_original_metric_differences'][metric], 'CI lower': h_ci['lower_95_percentile'], 'CI upper': h_ci['upper_95_percentile']},
            {'Domain': domain, 'Contrast': 'TabICLv2 minus original', 'Metric': DISPLAY_METRICS[metric], 'Point difference': t['tabiclv2_minus_original_metric_differences'][metric], 'CI lower': t_ci['lower'], 'CI upper': t_ci['upper']},
        ])
secondary_uncertainty = pd.DataFrame(secondary_ci)
display(Markdown('### Paired differences versus the initially selected model'))
display(secondary_uncertainty.round(3).style.hide(axis='index').format({'Point difference': '{:+.3f}', 'CI lower': '{:+.3f}', 'CI upper': '{:+.3f}'}))
display(Markdown('A 95% interval containing zero does not provide reliable evidence of a difference for that metric. There is no stored direct paired interval for TabICLv2 versus the hybrid.'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for ax, domain in zip(axes, DOMAINS.values()):
    view = secondary[secondary['Domain'] == domain].sort_values('Balanced accuracy', ascending=True)
    color_map = {'Initially selected': '#59697A', 'Hybrid candidate': '#E0A526', 'Later experimental candidate': '#F44B93'}
    ax.barh(view['Model'], view['Balanced accuracy'], color=[color_map[s] for s in view['Model stage']])
    ax.set_title(domain)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Balanced accuracy')
fig.suptitle('Balanced-accuracy ranking on the common secondary cohort', fontsize=14)
fig.tight_layout()
plt.show()

display(Markdown('### Domain-by-domain justification'))
for domain in DOMAINS.values():
    view = secondary[secondary['Domain'] == domain]
    best_ba = view.loc[view['Balanced accuracy'].idxmax()]
    best_ap = view.loc[view['PR-AUC'].idxmax()]
    max_sens = view['Sensitivity'].max()
    sensitivity_leaders = ', '.join(view.loc[np.isclose(view['Sensitivity'], max_sens), 'Model'])
    display(Markdown(
        f"**{domain}.** Highest balanced accuracy: **{best_ba['Model']}** ({best_ba['Balanced accuracy']:.3f}). "
        f"Highest PR-AUC: **{best_ap['Model']}** ({best_ap['PR-AUC']:.3f}). "
        f"Highest sensitivity: **{sensitivity_leaders}** ({max_sens:.3f})."
    ))

## 5. Unbiased synthesis framework

A model should not be called better merely because it has the highest value in one table. A defensible conclusion should consider:

- development PR-AUC and its repeat stability;
- the sensitivity–specificity trade-off at the frozen domain threshold;
- probability quality through Brier score and calibration evidence;
- paired uncertainty when it is available;
- consistency across domains rather than extrapolation from one domain;
- whether the evidence is development, original locked-test, secondary exploratory, or external validation; and
- implementation, explainability and governance constraints.

The most supportable wording is therefore **metric- and domain-specific**: for example, a method may have a higher PR-AUC point estimate for one domain while another has higher sensitivity. TabICLv2 and the hybrids remain exploratory unless supported by new untouched or external validation.

### Suggested supervisor conclusion

> On the common 125-participant secondary cohort, TabICLv2 produced the highest PR-AUC point estimate in all three domains. Balanced-accuracy results were mixed: the equal-weight hybrid ranked first for Communication, whereas TabICLv2 ranked first for Cognitive and Social-emotional development. Sensitivity also showed domain-specific trade-offs. Paired uncertainty generally did not establish reliable overall superiority, and the cohort was previously evaluated. These findings support TabICLv2 as a promising later experimental candidate, not as a conclusively superior or independently validated final model.

## 6. Reporting limitations

1. The development cohort contains 500 participants, with relatively few positive outcomes per domain.
2. Repeated cross-validation estimates are correlated because the same participants appear across repeats.
3. Later isolated candidate reports do not provide every possible participant-aligned pairwise comparison.
4. The 125-participant cohort was previously examined; later hybrid and TabICLv2 evaluations on it are secondary and exploratory.
5. No external validation is represented here. Generalisability to other populations or settings is unestablished.
6. Multiple models, domains and metrics were inspected; apparent differences may reflect sampling variation.
7. These models estimate research outcomes defined from later domain scores. They do not diagnose developmental conditions.
8. The notebook summarizes stored evidence and deliberately does not recompute analyses from sensitive participant-level data.